In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA
from tqdm import tqdm
from pca import pca
from sklearn.linear_model import LogisticRegression
import os
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
import matplotlib.pyplot as plt
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/')

In [2]:
hr = pd.read_csv("HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [3]:
X, y = hr.drop('left', axis = 1), hr['left']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26, stratify = y)

In [5]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
trns = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                              remainder = 'passthrough', verbose_feature_names_out=False)

trns = trns.set_output(transform = 'pandas')
scaler = StandardScaler()
svm = SVC()
comps = np.arange(2, 16)
scores = []
for c in tqdm(comps):
    prcomp = PCA(n_components = c).set_output(transform = 'pandas')
    pipe = Pipeline([('OHE', trns), ('SCL', scaler), ('PCA', prcomp), ('SVM', svm)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    scores.append([c, accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns = ['comps', 'score'])
df_scores.sort_values(['score', 'comps'], ascending = [False, True])

100%|███████████████████████████████████████████| 14/14 [00:24<00:00,  1.75s/it]


,comps,score
13,15,0.942654
12,14,0.941543
11,13,0.932874
10,12,0.930429
9,11,0.929318
7,9,0.926873
8,10,0.926650
5,7,0.926428
6,8,0.926206
4,6,0.925984


In [6]:
X, y = hr.drop('left', axis = 1), hr['left']

In [7]:
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)

In [8]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
trns = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                              remainder = 'passthrough', verbose_feature_names_out=False)
pipe = Pipeline([('OHE', trns), ('SCL', scaler), ('PCA', prcomp), ('SVM', svm)])
params = {
    'PCA__n_components' : np.arange(2,19),
    'SVM__kernel' : ['linear','rbf']
}

gcv = GridSearchCV(estimator=pipe,param_grid=params,cv=kfold,n_jobs=-1)
gcv.fit(X,y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=26, shuffle=True),
             estimator=Pipeline(steps=[('OHE',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('onehotencoder',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7f74abbff130>)],
                                                          verbose_feature_names_out=False)),
                                       ('SCL', StandardScaler()),
                                       ('PCA', PCA(n_components=15)),
                                       ('SVM', SVC())]),
             n_jobs=-1,
             param_grid={'PCA__n_components': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18]),
                         'SVM__kernel': ['linear', 'rbf']})

In [9]:
gcv.best_params_,gcv.best_score_

({'PCA__n_components': 18, 'SVM__kernel': 'rbf'}, 0.950383461153718)

In [10]:
pd.DataFrame(gcv.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_PCA__n_components,param_SVM__kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,4.479059,0.141619,0.606716,0.012114,2,linear,"{'PCA__n_components': 2, 'SVM__kernel': 'linear'}",0.761921,0.761921,0.762254,0.762254,0.762254,0.762121,0.000163,21
1,2.976267,0.071583,0.830772,0.012439,2,rbf,"{'PCA__n_components': 2, 'SVM__kernel': 'rbf'}",0.905635,0.903968,0.917639,0.903968,0.905969,0.907436,0.005168,16
2,4.612014,0.119301,0.642635,0.034603,3,linear,"{'PCA__n_components': 3, 'SVM__kernel': 'linear'}",0.761921,0.761921,0.762254,0.762254,0.762254,0.762121,0.000163,21
3,2.996712,0.065589,0.907579,0.022273,3,rbf,"{'PCA__n_components': 3, 'SVM__kernel': 'rbf'}",0.907969,0.902301,0.915972,0.903968,0.905635,0.907169,0.004784,17
4,4.719893,0.145957,0.657367,0.006706,4,linear,"{'PCA__n_components': 4, 'SVM__kernel': 'linear'}",0.761921,0.761921,0.762254,0.762254,0.762254,0.762121,0.000163,21
5,2.769049,0.130419,0.877041,0.027567,4,rbf,"{'PCA__n_components': 4, 'SVM__kernel': 'rbf'}",0.914305,0.912971,0.925642,0.912971,0.919973,0.917172,0.004962,15
6,4.745054,0.126264,0.679071,0.001886,5,linear,"{'PCA__n_components': 5, 'SVM__kernel': 'linear'}",0.761921,0.761921,0.762254,0.762254,0.762254,0.762121,0.000163,21
7,2.704867,0.085772,0.873114,0.011461,5,rbf,"{'PCA__n_components': 5, 'SVM__kernel': 'rbf'}",0.918640,0.915305,0.931977,0.917973,0.919973,0.920774,0.005805,14
8,5.647879,0.159469,0.686531,0.005195,6,linear,"{'PCA__n_components': 6, 'SVM__kernel': 'linear'}",0.761921,0.761921,0.762254,0.762254,0.762254,0.762121,0.000163,21
9,2.531231,0.086906,0.811198,0.026829,6,rbf,"{'PCA__n_components': 6, 'SVM__kernel': 'rbf'}",0.928643,0.925308,0.935979,0.921641,0.928976,0.928109,0.004747,13


In [14]:
lr = LogisticRegression()
pipe = Pipeline([('OHE', trns), ('SCL', scaler), ('PCA', prcomp), ('LR', lr)])
params = {
    'PCA__n_components' : np.arange(2,19),
    'LR__C' : np.linspace(0.01, 10, 20)
}

gcv = GridSearchCV(estimator=pipe,param_grid=params,cv=kfold,n_jobs=-1)
gcv.fit(X,y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=26, shuffle=True),
             estimator=Pipeline(steps=[('OHE',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('onehotencoder',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7f74abbff130>)],
                                                          verbose_feature_names_out=False)),
                                       ('SCL'...
                                       ('PCA', PCA(n_components=15)),
                                       ('LR', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'LR__C': array([ 0.01      ,  0.53578947,  1.06157895,  1.58736842,  2.11315789,
        2.63894737,  3.16473684,  3.69052632,  4.21631579,  4.74210526,
        5.26789474,  5.79368421,  6.31947368,  6.84526316,  7.37105263,
        7.89684211,  8.42263158,  8.94842105,  9.47421053, 10.        ]),
                         'PCA__n_components': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])})

In [15]:
gcv.best_params_, gcv.best_score_

({'LR__C': 4.2163157894736845, 'PCA__n_components': 18}, 0.7909303101033678)